In [4]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(PROJECT_ROOT)

In [56]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score
from utils import *

In [6]:
df_ = pd.read_pickle('data/panel/cleaned_data_2.pkl').drop_duplicates()

In [37]:
missing_values = show_missing_values(df_)
missing_values

,Column Name,Min,Max,n Unique,NaN count,NaN percentage,dtype
S. No.,,,,,,,
1,obid,18948200,1925607441,1577083,0,0.0%,int64
2,plz,1029,99478,3625,0,0.0%,int64
3,mietekalt,148.0,2690.0,76408,0,0.0%,float64
4,wohnflaeche,18.3,165.0,13535,0,0.0%,float64
5,etage,-1,45,47,0,0.0%,int64
6,zimmeranzahl,0.0,10.0,55,0,0.0%,float64
7,schlafzimmer,0.0,8.0,9,0,0.0%,float64
8,badezimmer,0.0,5.0,6,0,0.0%,float64
9,aufzug,0,1,2,0,0.0%,int64


In [40]:
missing_values.to_excel('n_unique.xlsx', index=False)

In [41]:
df = df_.copy()
df.head()

,obid,plz,mietekalt,wohnflaeche,etage,zimmeranzahl,schlafzimmer,badezimmer,aufzug,balkon,...,haustier_erlaubt,heizungsart,kategorie_Wohnung,objektzustand,blid,rent_sqm,is_schlafzimmer_imputed,is_parkplatz_imputed,year,month
81,41534430,22587,918.00,114.700000,1,4.0,1.0,1.0,0,1,...,By arrangement,Central heating,Not specified,Completely renovated,Hamburg,8.003488,False,True,2007,5
94,42410574,20251,374.33,48.000000,1,2.0,1.0,1.0,0,1,...,By arrangement,Not specified,Flat,Well-kept,Hamburg,7.798541,False,True,2007,7
127,38404913,22303,565.60,74.000000,4,3.0,1.0,1.0,0,0,...,No,Central heating,Flat,Well-kept,Hamburg,7.643243,False,True,2007,6
162,41026870,22765,284.00,40.410000,4,2.0,1.0,1.0,1,1,...,No,Central heating,Attic flat,Like new,Hamburg,7.027964,False,True,2007,3
171,36771732,20357,373.00,55.439999,2,2.0,1.0,1.0,0,0,...,By arrangement,Self-contained central heating,Flat,Not specified,Hamburg,6.727994,False,True,2007,7


In [42]:
df.shape

(2067056, 27)

In [43]:
var_types = {
    "obid": "kat",
    "plz": "kat",
    "mietekalt": "cont",
    "wohnflaeche": "cont",
    "etage": "kat",
    "zimmeranzahl": "nom",
    "schlafzimmer": "nom",
    "badezimmer": "nom",
    "aufzug": "bin",
    "balkon": "bin",
    "einbaukueche": "bin",
    "foerderung": "kat",
    "gaestewc": "bin",
    "garten": "bin",
    "keller": "bin",
    "parkplatz": "bin",
    "ausstattung": "kat",
    "haustier_erlaubt": "kat",
    "heizungsart": "kat",
    "kategorie_Wohnung": "kat",
    "objektzustand": "kat",
    "blid": "kat",
    "rent_sqm": "cont",
    "year": "nom",
    "month": "nom"
}

In [44]:
nom_cols = [col for col, t in var_types.items() if t == "nom"]
df[nom_cols] = df[nom_cols].apply(pd.to_numeric)

In [45]:
df.head()

,obid,plz,mietekalt,wohnflaeche,etage,zimmeranzahl,schlafzimmer,badezimmer,aufzug,balkon,...,haustier_erlaubt,heizungsart,kategorie_Wohnung,objektzustand,blid,rent_sqm,is_schlafzimmer_imputed,is_parkplatz_imputed,year,month
81,41534430,22587,918.00,114.700000,1,4.0,1.0,1.0,0,1,...,By arrangement,Central heating,Not specified,Completely renovated,Hamburg,8.003488,False,True,2007,5
94,42410574,20251,374.33,48.000000,1,2.0,1.0,1.0,0,1,...,By arrangement,Not specified,Flat,Well-kept,Hamburg,7.798541,False,True,2007,7
127,38404913,22303,565.60,74.000000,4,3.0,1.0,1.0,0,0,...,No,Central heating,Flat,Well-kept,Hamburg,7.643243,False,True,2007,6
162,41026870,22765,284.00,40.410000,4,2.0,1.0,1.0,1,1,...,No,Central heating,Attic flat,Like new,Hamburg,7.027964,False,True,2007,3
171,36771732,20357,373.00,55.439999,2,2.0,1.0,1.0,0,0,...,By arrangement,Self-contained central heating,Flat,Not specified,Hamburg,6.727994,False,True,2007,7


In [46]:
X = df.drop(columns=['obid', 'mietekalt', 'plz'])
y = df['mietekalt']

In [47]:
bin_cols = [col for col, t in var_types.items() if t == "bin"]
X[bin_cols] = X[bin_cols].astype(int)

In [48]:
kat_cols = [col for col, t in var_types.items() if t == "kat" and col in X.columns]
X = pd.get_dummies(X, columns=kat_cols, drop_first=True)

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [52]:
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=50, random_state=42)

In [55]:
y_pred = model.predict(X_test)
print("MAE:", mean_squared_error(y_test, y_pred))
print("MSE:", mean_absolute_error(y_test, y_pred))
print("R² Score:", r2_score(y_test, y_pred))

MAE: 0.8298562834978176
MSE: 0.23775873898193783
R² Score: 0.9999949361088657


In [58]:
X.shape

(2067056, 116)

In [59]:
2067056 * 216

446484096

In [60]:
import joblib

joblib.dump(model, 'model.pkl')

['model.pkl']

In [61]:
df['predicted_mietekalt'] = model.predict(X)

In [62]:
df.head()

,obid,plz,mietekalt,wohnflaeche,etage,zimmeranzahl,schlafzimmer,badezimmer,aufzug,balkon,...,heizungsart,kategorie_Wohnung,objektzustand,blid,rent_sqm,is_schlafzimmer_imputed,is_parkplatz_imputed,year,month,predicted_mietekalt
81,41534430,22587,918.00,114.700000,1,4.0,1.0,1.0,0,1,...,Central heating,Not specified,Completely renovated,Hamburg,8.003488,False,True,2007,5,917.9904
94,42410574,20251,374.33,48.000000,1,2.0,1.0,1.0,0,1,...,Not specified,Flat,Well-kept,Hamburg,7.798541,False,True,2007,7,374.5828
127,38404913,22303,565.60,74.000000,4,3.0,1.0,1.0,0,0,...,Central heating,Flat,Well-kept,Hamburg,7.643243,False,True,2007,6,565.8704
162,41026870,22765,284.00,40.410000,4,2.0,1.0,1.0,1,1,...,Central heating,Attic flat,Like new,Hamburg,7.027964,False,True,2007,3,284.0818
171,36771732,20357,373.00,55.439999,2,2.0,1.0,1.0,0,0,...,Self-contained central heating,Flat,Not specified,Hamburg,6.727994,False,True,2007,7,372.9298


In [64]:
df['difference'] = df['predicted_mietekalt'] - df['mietekalt']
df['difference'].describe()

count    2.067056e+06
mean    -3.152914e-03
std      5.285220e-01
min     -1.156054e+02
25%     -6.600000e-03
50%      0.000000e+00
75%      0.000000e+00
max      4.951600e+01
Name: difference, dtype: float64

In [65]:
df['difference'].describe().round(2)

count    2067056.00
mean          -0.00
std            0.53
min         -115.61
25%           -0.01
50%            0.00
75%            0.00
max           49.52
Name: difference, dtype: float64

In [70]:
df.to_csv('predicted_data_2.csv', index=False)